# 05 - FAISS Retrieval Engine

This notebook demonstrates the `RecipeRetriever` class, which combines ingredient vectors and mood vectors into a weighted query, searches a FAISS index of 13,482 recipe embeddings, and returns results with explainability metadata.

**Query vector construction:**
- If both ingredients and mood are present: `0.7 * ingredient_vec + 0.3 * mood_vec`
- If only one signal is present: weight = 1.0
- The combined vector is L2-normalized before searching with inner product (cosine similarity on unit vectors)

**Design decision:** Constraints (time, effort) are echoed in the explanation but do NOT hard-filter results, because the Epicurious dataset lacks structured time/effort fields.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))

from pantry.retrieval import RecipeRetriever

INDEX_PATH = "data/processed/recipes.index"
RECIPES_PATH = "data/processed/recipes.json"

retriever = RecipeRetriever(INDEX_PATH, RECIPES_PATH)
print(f"Index size: {retriever._index.ntotal} vectors")
print(f"Recipe count: {len(retriever._recipes)}")

Index size: 13482 vectors
Recipe count: 13482


## 1. Ingredient-only search

Search using only ingredient tokens, no mood signal.

In [2]:
results = retriever.search(
    ingredient_tokens=["chicken", "garlic", "lemon"],
    mood_vector=None,
    constraints={},
    top_k=5,
)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score: {r['score']:.4f}) ---")
    print(f"Title: {r['title']}")
    print(f"Matched ingredients: {r['explanation']['matched_ingredients']}")
    print(f"Mood match: {r['explanation']['mood_match']}")


--- Result 1 (score: 0.9518) ---
Title: Grilled Chicken and Shrimp Kebabs with Lemon and Garlic
Matched ingredients: ['chicken', 'garlic', 'lemon']
Mood match: False

--- Result 2 (score: 0.9483) ---
Title: Pumpkin Shrimp Curry
Matched ingredients: ['garlic']
Mood match: False

--- Result 3 (score: 0.9474) ---
Title: Mulligatawny Soup
Matched ingredients: ['chicken', 'garlic', 'lemon']
Mood match: False

--- Result 4 (score: 0.9455) ---
Title: Chicken and Cashew Stir-Fry
Matched ingredients: ['chicken', 'garlic']
Mood match: False

--- Result 5 (score: 0.9448) ---
Title: Chicken Tortilla Soup
Matched ingredients: ['chicken', 'garlic']
Mood match: False


## 2. Mood-only search

Search using only a mood vector (no ingredients). We use `MoodEmbedder` to get the vector.

In [3]:
from pantry.mood_embedder import MoodEmbedder

mood_embedder = MoodEmbedder()
mood_result = mood_embedder.embed("something cozy and warm for a cold evening")
print(f"Mood tokens detected: {mood_result['mood_tokens']}")
print(f"Mood vector present: {mood_result['vector'] is not None}")

results = retriever.search(
    ingredient_tokens=[],
    mood_vector=mood_result["vector"],
    constraints={},
    top_k=5,
)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score: {r['score']:.4f}) ---")
    print(f"Title: {r['title']}")
    print(f"Mood match: {r['explanation']['mood_match']}")

Mood tokens detected: ['cozy', 'warm', 'cold']
Mood vector present: True

--- Result 1 (score: 0.7106) ---
Title: Amaretti Tiramisu
Mood match: True

--- Result 2 (score: 0.7045) ---
Title: The Stout Diplomat
Mood match: True

--- Result 3 (score: 0.6974) ---
Title: Southern Comfort Champagne Cocktail
Mood match: True

--- Result 4 (score: 0.6970) ---
Title: Bubble-Top Brioches
Mood match: True

--- Result 5 (score: 0.6950) ---
Title: Fried Mozzarella Balls
Mood match: True


## 3. Combined search: ingredients + mood

When both signals are present, the query vector is a weighted blend: `0.7 * ingredients + 0.3 * mood`. This lets the mood nudge results toward a certain style without overwhelming the ingredient signal.

In [4]:
mood_result = mood_embedder.embed("something elegant and fancy")
print(f"Mood tokens: {mood_result['mood_tokens']}")

results = retriever.search(
    ingredient_tokens=["salmon", "dill", "cream"],
    mood_vector=mood_result["vector"],
    constraints={"time": "under 30 min", "effort": "medium"},
    top_k=5,
)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score: {r['score']:.4f}) ---")
    print(f"Title: {r['title']}")
    print(f"Matched ingredients: {r['explanation']['matched_ingredients']}")
    print(f"Mood match: {r['explanation']['mood_match']}")
    print(f"Constraints noted: {r['explanation']['constraints_met']}")

Mood tokens: ['elegant', 'fancy']

--- Result 1 (score: 0.8902) ---
Title: Simple Poached Salmon
Matched ingredients: ['salmon', 'dill']
Mood match: True
Constraints noted: {'time': 'under 30 min', 'effort': 'medium'}

--- Result 2 (score: 0.8877) ---
Title: Citrus-and-Dill Gravlax
Matched ingredients: ['salmon', 'dill', 'cream']
Mood match: True
Constraints noted: {'time': 'under 30 min', 'effort': 'medium'}

--- Result 3 (score: 0.8772) ---
Title: Avo and Egg
Matched ingredients: []
Mood match: True
Constraints noted: {'time': 'under 30 min', 'effort': 'medium'}

--- Result 4 (score: 0.8731) ---
Title: Smoked Trout, Crème Fraîche & Pickled Onion
Matched ingredients: ['dill']
Mood match: True
Constraints noted: {'time': 'under 30 min', 'effort': 'medium'}

--- Result 5 (score: 0.8728) ---
Title: Caviar on Potato with Creamy Champagne Dressing
Matched ingredients: ['cream']
Mood match: True
Constraints noted: {'time': 'under 30 min', 'effort': 'medium'}


## 4. Detailed explanation for top result

Let's inspect the full result dict to see every field available to downstream consumers.

In [5]:
import json

top = results[0]
# Pretty-print everything except the full instructions (can be long)
display_copy = {**top, "instructions": top["instructions"][:200] + "..."}
print(json.dumps(display_copy, indent=2, default=str))

{
  "id": 11066,
  "title": "Simple Poached Salmon",
  "ingredients": [
    "water",
    "dry white wine",
    "yellow onion slice",
    "lemon slice",
    "dill",
    "salt",
    "salmon fillet"
  ],
  "instructions": "Combine the water and wine in the slow cooker and heat on high for 20 to 30 minutes. Add the onion, lemon, dill, salt, and salmon.\nCover and cook on high for about 20 minutes, until the salmon is opaq...",
  "image_name": "simple-poached-salmon-237292",
  "score": 0.8902490139007568,
  "explanation": {
    "matched_ingredients": [
      "salmon",
      "dill"
    ],
    "mood_match": true,
    "constraints_met": {
      "time": "under 30 min",
      "effort": "medium"
    }
  }
}


## 5. Discussion

**Query vector weighting (0.7 / 0.3):** The 70-30 split prioritises concrete ingredient matches while still allowing mood to steer the ranking. This ratio was chosen because users typically care more about using specific ingredients than about abstract mood, but mood adds meaningful differentiation when ingredient overlap is similar across candidates.

**Over-fetching (3x):** We retrieve `top_k * 3` candidates from FAISS so that downstream post-processing (e.g. future hard-filtering on structured fields) has room to discard entries while still returning `top_k` results.

**Constraint handling:** The Epicurious dataset does not include structured time or effort metadata, so constraints are passed through to the explanation dict for transparency but do not filter results. A future enhancement could use an LLM or regex-based heuristic to estimate prep time from instruction text.